# Module 02: Offline Store & Training

## What You'll Learn

- How the offline store provides historical features for model training
- Point-in-time correctness and why it matters
- Entity DataFrames and temporal joins
- Different offline store backends (PostgreSQL, DuckDB, Spark)
- Creating training datasets with `get_historical_features()`

## Prerequisites

- Completed Module 01 (Core Concepts)
- RHOAI workbench with Feast installed
- PostgreSQL instance accessible (operator-managed or standalone)

---

> **🗺️ DATA STRATEGY**: The offline store is where Feast provides training-serving consistency for **predictive AI** workloads (Pillar 3). Every model trained on Feast features gets point-in-time correct data — eliminating the #1 cause of model degradation in production.

## Point-in-Time Correctness

The most important concept in offline feature retrieval is **point-in-time correctness**.

When training a model, you need features *as they were at the time of each training example* — not the latest values. Otherwise you introduce **data leakage** (using future information to predict the past).

```
Timeline:
  Jan 1: credit_score = 720
  Feb 1: credit_score = 680  (missed payment)
  Mar 1: credit_score = 650  (another miss)
  Apr 1: credit_score = 700  (recovered)

Training example at Feb 15:
  ✅ Should get: credit_score = 680 (latest as of Feb 15)
  ❌ Should NOT get: credit_score = 700 (April value = data leakage)
```

Feast handles this automatically via temporal joins on `event_timestamp`.

> **⚠️ GAP**: Point-in-time correctness is *supported* but not *architecturally enforced*. Documentation confusion between `event_timestamp` and `created_timestamp` can lead to subtle bugs. SQL type inference bugs can silently corrupt temporal joins. This is a known P1 gap — Chronon (competitor) enforces correctness by design.

## Setup: Feature Store with PostgreSQL Offline Store

On RHOAI, the typical offline store is PostgreSQL (deployed as part of the FeatureStore CR or as a standalone instance).

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os

# Generate time-series feature data to demonstrate point-in-time joins
np.random.seed(42)

# Create customer features that change over time
records = []
base_time = datetime(2024, 1, 1)

for customer_id in range(1, 51):
    credit_score = np.random.randint(600, 800)
    for week in range(52):  # One year of weekly updates
        ts = base_time + timedelta(weeks=week)
        # Score drifts randomly
        credit_score += np.random.randint(-20, 20)
        credit_score = max(300, min(850, credit_score))
        
        records.append({
            "customer_id": customer_id,
            "event_timestamp": ts,
            "credit_score": credit_score,
            "debt_to_income_ratio": round(np.random.uniform(0.1, 0.9), 3),
            "num_open_accounts": np.random.randint(1, 15),
            "total_credit_utilization": round(np.random.uniform(0.0, 1.0), 3),
        })

features_df = pd.DataFrame(records)
print(f"Generated {len(features_df)} feature records")
print(f"Date range: {features_df['event_timestamp'].min()} to {features_df['event_timestamp'].max()}")
features_df.head(10)

In [ ]:
# Save as parquet for the file-based offline store
os.makedirs("data", exist_ok=True)
features_df.to_parquet("data/credit_features_timeseries.parquet")
print("Saved time-series feature data")

In [ ]:
from feast import Entity, FeatureView, Field, FileSource, FeatureStore
from feast.types import Float32, Int64

# Define feature store components
customer = Entity(name="customer", join_keys=["customer_id"])

credit_source = FileSource(
    name="credit_timeseries_source",
    path=os.path.abspath("data/credit_features_timeseries.parquet"),
    timestamp_field="event_timestamp",
)

credit_fv = FeatureView(
    name="credit_history",
    entities=[customer],
    ttl=timedelta(weeks=2),
    schema=[
        Field(name="credit_score", dtype=Int64),
        Field(name="debt_to_income_ratio", dtype=Float32),
        Field(name="num_open_accounts", dtype=Int64),
        Field(name="total_credit_utilization", dtype=Float32),
    ],
    source=credit_source,
)

In [ ]:
import os
os.makedirs("feature_repo", exist_ok=True)

with open("feature_repo/feature_store.yaml", "w") as f:
    f.write("""project: offline_store_demo
provider: local
registry:
  registry_type: sql
  path: sqlite:///data/registry.db
online_store:
  type: sqlite
  path: data/online_store.db
offline_store:
  type: duckdb
entity_key_serialization_version: 3
""")
print("Written: feature_repo/feature_store.yaml")

In [ ]:
os.makedirs("feature_repo", exist_ok=True)
store = FeatureStore(repo_path="feature_repo")
store.apply([customer, credit_source, credit_fv])
print("✅ Feature store applied")

## Entity DataFrames

An **Entity DataFrame** is the "request" you send to the offline store. It specifies:
- Which entities (customers) you want features for
- At which point in time you want each entity's features

This is typically derived from your training labels (e.g., "customer X defaulted on date Y").

In [ ]:
# Simulate training labels: "did customer default in this month?"
# Each row = one training example with a specific timestamp
entity_df = pd.DataFrame({
    "customer_id": [1, 1, 2, 3, 5, 10, 15, 20, 25, 30],
    "event_timestamp": [
        datetime(2024, 3, 1),   # Customer 1 in March
        datetime(2024, 6, 1),   # Customer 1 in June (different point in time!)
        datetime(2024, 4, 15),  # Customer 2 in April
        datetime(2024, 5, 1),   # Customer 3 in May
        datetime(2024, 7, 1),
        datetime(2024, 8, 1),
        datetime(2024, 9, 1),
        datetime(2024, 10, 1),
        datetime(2024, 11, 1),
        datetime(2024, 12, 1),
    ],
    "label_defaulted": [0, 0, 1, 0, 0, 1, 0, 0, 1, 0],  # Training labels
})

print("Entity DataFrame (training request):")
print("Each row asks: 'What were this customer's features AT THIS TIMESTAMP?'")
entity_df

## Historical Feature Retrieval

`get_historical_features()` performs point-in-time joins: for each row in the entity DataFrame, it finds the **most recent feature values as of that timestamp** (respecting TTL).

In [ ]:
# Retrieve historical features with point-in-time correctness
training_data = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "credit_history:credit_score",
        "credit_history:debt_to_income_ratio",
        "credit_history:num_open_accounts",
        "credit_history:total_credit_utilization",
    ],
).to_df()

print("Training dataset with point-in-time correct features:")
print("Notice: Customer 1 has DIFFERENT credit_score values for March vs June")
training_data

In [ ]:
# Demonstrate point-in-time correctness:
# Customer 1's features should differ between March and June
customer_1_rows = training_data[training_data["customer_id"] == 1]
print("Customer 1 features at two different points in time:")
print(customer_1_rows[["customer_id", "event_timestamp", "credit_score", "debt_to_income_ratio"]])
print("\n✅ Different values = point-in-time correctness working!")
print("   Each row reflects the state of the world AT THAT TIMESTAMP.")

## TTL (Time-to-Live) Behaviour

TTL defines how old a feature value can be before it's considered stale and returned as `NULL`.

Our feature view has `ttl=timedelta(weeks=2)`. If we request features at a timestamp where the most recent value is older than 2 weeks, we get `NULL`.

> **⚠️ GAP**: TTL expiry silently drops features rather than alerting. Users can get empty feature vectors without knowing their data went stale. There is no staleness alerting or materialization failure notification. This is a P0 gap — silent failures are the #1 client/partner complaint.

In [ ]:
# Request features at a time BEFORE any data exists (TTL will cause NULLs)
stale_entity_df = pd.DataFrame({
    "customer_id": [1, 2, 3],
    "event_timestamp": [
        datetime(2023, 6, 1),  # 6 months before our data starts
        datetime(2023, 6, 1),
        datetime(2023, 6, 1),
    ],
})

stale_result = store.get_historical_features(
    entity_df=stale_entity_df,
    features=["credit_history:credit_score"],
).to_df()

print("Requesting features from before data exists:")
print(stale_result)
print("\nNULL values = no data within TTL window. Silent failure.")

## PostgreSQL Offline Store (RHOAI Production Pattern)

On RHOAI, the production offline store is typically PostgreSQL. Here's how to configure it:

```yaml
# feature_store.yaml for PostgreSQL offline store
project: credit_scoring
provider: local
registry:
  registry_type: sql
  path: postgresql+psycopg://<user>:<pass>@<host>:5432/<db>
offline_store:
  type: postgres
  host: feast-postgresql.my-namespace.svc.cluster.local
  port: 5432
  database: feast
  db_schema: public
  user: feast
  password: ${FEAST_POSTGRES_PASSWORD}  # From K8s Secret
online_store:
  type: postgres
  host: feast-postgresql.my-namespace.svc.cluster.local
  port: 5432
  database: feast_online
  user: feast
  password: ${FEAST_POSTGRES_PASSWORD}
```

> **📍 RHOAI STATUS**: PostgreSQL can serve as offline store, online store, AND registry simultaneously — a single-database deployment pattern that simplifies operations. The Feast operator supports this.
>
> **⚠️ GAP**: The CRD uses inline Kubernetes Secrets for credentials, not platform-level connection objects. Workshop Decision #2 (Centralized Connection Auth) and Decision #4 (Credentials Auto-Mounting) would change this, but they are not yet delivered.

## Training a Model with Feast Features

Let's use the retrieved features to train a simple model — the complete predictive AI workflow.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report

# Prepare training data from Feast
feature_cols = ["credit_score", "debt_to_income_ratio", "num_open_accounts", "total_credit_utilization"]
X = training_data[feature_cols].fillna(0)
y = training_data["label_defaulted"]

print(f"Training with {len(X)} samples, {len(feature_cols)} features from Feast")
print(f"Features served by: credit_history FeatureView")
print(f"\nThis is the Feast value proposition:")
print(f"  - Same feature definitions will serve online inference (Module 03)")
print(f"  - No training-serving skew possible")

## Offline Store Backends Comparison

| Backend | Best For | RHOAI Status | Notes |
|---------|----------|-------------|-------|
| **PostgreSQL** | Production on RHOAI | GA | Operator-managed, dual-use (online+offline) |
| **DuckDB** | Local development, testing | Available | In-process, no server needed |
| **Spark** | Large-scale historical joins | Community plugin | Needs Spark Operator (GA in 3.5) |
| **Snowflake** | Cloud data warehouse integration | Community plugin | For customers with Snowflake |
| **BigQuery** | GCP customers | Official | — |
| **Redshift** | AWS customers | Official | — |

> **🗺️ DATA STRATEGY**: The compute engines (Ray, Spark) from Pillar 2 are consumed here — Feast's `get_historical_features()` can delegate heavy joins to Spark or Ray rather than running them locally. This is the Feast+Compute integration covered in Module 06.

## Key Takeaways

1. **Point-in-time joins** ensure training data reflects the world as it was, not as it is now
2. **Entity DataFrames** are the "request" — specify which entities at which timestamps
3. **TTL** controls staleness — values outside the window return NULL (silently!)
4. **PostgreSQL** is the production offline store on RHOAI
5. The same feature definitions serve both training (this module) and inference (Module 03)

## What's Next

- **Module 03**: Online Store — materialization, low-latency serving, `get_online_features()`
- **Module 06**: Ray Compute — using Ray as the compute engine for large-scale historical joins